# Analytic Tav multipole ladder

The fixed ladder formula predicts CMB multipoles:

$$\ell = \mathrm{round}(7 \cdot k \cdot \alpha + r), \quad r \in \{0, 2, 3, 6\}, \quad \alpha = 41.341$$

This notebook generates the ladder, compares it to reference peaks, and shows a digamma redshift step.

In [ ]:
import numpy as np

from tav_resonance import (
    TavResonanceConfig,
    analytic_tav_ladder,
    evaluate_ladder_vs_observed,
    geometric_anchor_check,
    photon_redshift_step,
)
from tav_resonance.anchors import REFERENCE_LADDER_PEAKS

## Anchor validation

In [ ]:
geometric_anchor_check()

## Generate the analytic ladder

In [ ]:
cfg = TavResonanceConfig.from_defaults()
predicted = analytic_tav_ladder(k_max=cfg.ladder_k_max, config=cfg)

print(f"Predicted multipoles ({len(predicted)} total):")
print(predicted[:16], "...")

## Compare against reference peaks

In [ ]:
report = evaluate_ladder_vs_observed(REFERENCE_LADDER_PEAKS, config=cfg)
print(f"α used (fixed): {report['alpha_used']}")
print(f"Peaks compared: {report['n_peaks_compared']}")
print(f"Mean |residual|: {report['mean_absolute_residual']:.2f}")
report["comparison_table"][:5]

## Digamma photon redshift step (Dynamic Refresh)

In [ ]:
for node in range(1, 6):
    step = photon_redshift_step(node)
    print(f"node {node}: ΔE/E = {step['delta_E_fraction']:.6f}")

## Optional: observed vs predicted residual plot

In [ ]:
try:
    import matplotlib.pyplot as plt

    table = report["comparison_table"]
    observed = [row["observed"] for row in table]
    predicted_ell = [row["predicted"] for row in table]
    residuals = [row["residual"] for row in table]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(observed, residuals, label="residual (obs − pred)")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Observed multipole ℓ")
    ax.set_ylabel("Residual")
    ax.set_title("Analytic Tav ladder vs reference peaks")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skip plot or install tav-resonance[viz]")